# Survey Responder Google Colab Quickstart

---
This Google Colab serves as an alternative approach to working with the [Survey Responder](https://github.com/DhruvKithany/SurveyResponder).

In this notebook, you'll learn how to:

- Install and run SurveyResponder

- Generate response datasets with different LLMs

- Customize personas and questions

- Analyze the generated data




## 1. Installation & Setup
---
First, let's install the Survey Responder and necessary tools to run it


In [ ]:
# Setup complete

In [ ]:
# Download the SurveyResponder script and required files
!git clone https://github.com/DhruvKithany/SurveyResponder.git
%cd SurveyResponder
!apt-get update
!apt-get install -y zstd
# Install Ollama to use LLM's locally
!curl -fsSL https://ollama.com/install.sh | sh


## 2. Start the Ollama Server
---
Next, we will start the Ollama server in order to pull LLM's and generate responses

You can learn more about Ollama CLI commands by visiting:
https://github.com/ollama/ollama/tree/main/docs

In [ ]:
# Start the Ollama server in the background so it can handle LLM requests
# "nohup" keeps it running even after this cell finishes
import subprocess, time

# Kill and restart ollama with correct env
subprocess.Popen([
    "bash", "-c",
    """
    pkill ollama
    sleep 2
    export CUDA_VISIBLE_DEVICES=0
    export OLLAMA_FLASH_ATTENTION=1
    export OLLAMA_KV_CACHE_TYPE=q4_0
    export OLLAMA_KEEP_ALIVE=-1
    ollama serve > /tmp/ollama.log 2>&1 &
    """
])

time.sleep(5)
#!nohup ollama serve &

## You can learn more about Ollama CLI commands by visiting:
#https://github.com/ollama/ollama/tree/main/docs


## 3. Pull a Language Model from Ollama
---
Download the LLM you would like to use for generating responses

In [ ]:
# Pull the latest gemma3n model (will take a few minutes)
!ollama pull llama3-chatqa:8b

In [ ]:
# Check the models you have pulled
!ollama list

In [ ]:
# Remove a model (if needed)
!ollama rm lfm2:24b

In [ ]:
# If you have trouble getting responses later, run the command below to confirm the local model server is working and accepting requests.
!curl -X POST http://localhost:11434/api/generate -d '{"model":"gemma3n:latest","prompt":"Expectation","stream":false}'

## 4. Generate Survey Responses
---
Now, we will instantiate and run the Survey Responder to generate LLM responses


SurveyResponder Parameters
- `questions_path`: Path to the text file containing survey questions (one per line).

- `persona_path`: Path to the JSON file that defines the persona traits and their prompt descriptions.

- `model_name`: The name of the LLM model to use (must be pulled using ollama pull first).

- `response_options`: List of Likert-scale response options to simulate

- `num_responses`: Number of synthetic personas and responses to generate. Default is 100.

- `temperature`: Controls the randomness/creativity of the LLM output (higher = more diverse responses).

- `base_url`: The endpoint where Ollama is serving the model API (localhost by default for local models).

In [ ]:
# Import the SurveyResponder }to be instantiated
from SurveyResponder import SurveyResponder

In [ ]:
import os

# === T4 OPTIMIZED SETTINGS ===
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_KV_CACHE_TYPE"] = "q4_0"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_CONTEXT_LENGTH"] = "8192"
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("✅ T4-optimized environment set")

In [ ]:

from google.colab import drive
import os
import pandas as pd

# Mount Google Drive for auto-backups
print('Mounting Google Drive...')
drive.mount('/content/drive')

responder = SurveyResponder(
      questions_path="prca_prca_questions.json",
      persona_path="persona.json",
      model_name="llama3-chatqa:8b",
            num_responses=10,
      temperature=1.0,
      base_url="http://localhost:11434/api/generate"
  )

# Output directly to Google Drive
base_drive_dir = '/content/drive/MyDrive/SurveyResponses'
model_safe = responder.model_name.replace(':', '_')
run_folder_name = f"run_{model_safe}_temp_{responder.temperature}"
run_dir = os.path.join(base_drive_dir, run_folder_name)
os.makedirs(run_dir, exist_ok=True)

output_file = os.path.join(run_dir, "llama3-chatqa.csv")
print(f"
Generating responses and saving continuously to:
{run_dir}")
responder.run_write(output_file)

In [ ]:
# This generates 100 synthetic personas and their responses to the questions listed in `prca_prca_questions.json`. Results are saved to `results.csv`.

from google.colab import drive
import os
import pandas as pd

# Mount Google Drive for auto-backups
print('Mounting Google Drive...')
drive.mount('/content/drive')

responder = SurveyResponder(
    questions_path="prca_prca_questions.json",
    persona_path="persona.json",
    model_name="mistral-nemo:12b",
        num_responses=10,
    temperature=1.0,
    base_url="http://localhost:11434/api/generate"
  )
f_name=f"mistral_nemo{5}.csv"


# Output directly to Google Drive
base_drive_dir = '/content/drive/MyDrive/SurveyResponses'
model_safe = responder.model_name.replace(':', '_')
run_folder_name = f"run_{model_safe}_temp_{responder.temperature}"
run_dir = os.path.join(base_drive_dir, run_folder_name)
os.makedirs(run_dir, exist_ok=True)

output_file = os.path.join(run_dir, f_name)
print(f"
Generating responses and saving continuously to:
{run_dir}")
df = responder.run_write(output_file)

In [ ]:
!find . -type f -name '*temp*' -delete

In [ ]:

import zipfile
import os
zip_name = "all_temp2_files.zip"
print("Packaging files into a zip archive...")
with zipfile.ZipFile(zip_name, 'w') as zipf:
    for i in np.arange(0.1, 2.05, 0.05):
        f_name = f"{i:.2f}_temp.csv"
        if os.path.exists(f_name):
            zipf.write(f_name)

print(f"Success! Created {zip_name}")

In [ ]:
import pandas as pd
def clean_csv(file_name: str):
    df = pd.read_csv(file_name)
    dd = {
        "Strongly Disagree": 1,
        "Disagree": 2,
        "Neutral": 3,
        "Agree": 4,
        "Strongly Agree": 5,
        "Never": 1,
        "Rarely": 2,
        "Sometimes": 3,
        "Often": 4,
        "Always": 5
    }
    for col in df.columns:
        if col.startswith("Q"):
            df[col] = df[col].map(dd)
    new_file_name = "cleaned_" + file_name
    df.to_csv(new_file_name,index=False)
    return f"Success Cleaning! Saved to {new_file_name} "


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("results_4.csv")
cols = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8', 'Q9', 'Q10', 'Q11', 'Q12']


bins = [0, 20, 40, 60, 80, 100]
labels = [
   1,2,3,4,5
]

# Apply the change to all specified columns at once
for col in cols:
    df[col] = pd.cut(df[col], bins=bins, labels=labels, include_lowest=True)

print(df.head())
df.to_csv("cleaned5.csv")

## 5. View Output
---
SurveyResponder automatically generates the following files when you run responder.run_write("results.csv"):

`results.csv`
- A table of simulated responses.

- Each row is a simulated persona.

- Columns include :

  - Demographic and personality traits

  - Likert-style answers for each question

`results_params.json`
- Stores metadata and settings used to generate the responses.

- Includes model name, temperature, number of responses, etc.

- Ensures your results are reproducible

In [ ]:
# To preview results.csv
import pandas as pd
df.head() # Or just "df" to view the entire dataframe

In [ ]:
# To preview results_params.json
import json

with open("results_params.json") as f:
    params = json.load(f)

params

## Customization Options
---
Below are a few examples of ways to customize and tailor the Survey Responder for specific use cases:

### Changing LLM Models

To test how responses differ among LLM models, you can change the LLM by pulling it from Ollama

A full list of available LLM's are found here: https://ollama.com/library

In [ ]:
# Example: pull mistral and use it in the responder
!ollama pull mistral:latest

from SurveyResponder import SurveyResponder
responder = SurveyResponder(
    questions_path="prca_prca_questions.json",
    persona_path="persona.json",
    model_name="mistral:latest", # Changed to mistral
        num_responses=100,
    temperature=1.0,
    base_url="http://localhost:11434/api/generate"
)

### Changing Response Options
The default likert scale can be changed to more accurately fit specific questions and personas, and it can be done via the following:



In [ ]:
responder = SurveyResponder(
    questions_path="prca_prca_questions.json",
    persona_path="persona.json",
    model_name="mistral:latest",
    # Changed to 4 point likert scale
    num_responses=100,
    temperature=1.0,
    base_url="http://localhost:11434/api/generate"
)


### Editing Questions and Personas

SurveyResponder uses two input files:

- `prca_prca_questions.json` — plain text, one survey question per line.
- `persona.json` — a dictionary of traits where each key becomes a column and each value is a list of `[value, description]` pairs.

You can edit these files manually in a file browser, text editor, or like this:

In [ ]:
# Add a new question to prca_prca_questions.json
with open("prca_prca_questions.json", "a") as f:
    f.write("\nI feel confident solving programming problems.")

# Add a new trait to persona.json
import json

with open("persona.json", "r") as f:
    personas = json.load(f)

# Add a new student status trait
personas["student_status"] = personas.get("student_status", [])
personas["student_status"].append(["full-time", "who is a full-time student"])

# Save the changes
with open("persona.json", "w") as f:
    json.dump(personas, f, indent=2)


### Preview Personas and Prompts

SurveyResponder allows you to preview generated personas and the survey prompts that will be sent to the language model.

This is useful for verifying that your `persona.json` is set up correctly and to better understand how the model interprets the question in context.

In [ ]:
# Create a SurveyResponder
responder = SurveyResponder()

# Generate a random persona description
response_options=["Never", "Rarely", "Often", "Always"]
# Output: "You are a someone who is multiracial, who is from a family whose members go to and do well in college..."

# Generate multiple personas
personas = responder.example_persona(npersonas=100)
questions= []
with open("prca_prca_questions.json",encoding='utf-8') as file:
    for line in file:
        questions.append(line[:-2])
len_response_options = 0
for i in range(len(response_options)):
  len_response_options+=len(response_options[i])
idx=0
tt=0
c_len=0
for i in range(len(personas)):
  for j in range(len(questions)):
    tt+=276+len(personas[i])+len(questions[j])+len_response_options

  c_len+=tt
  tt=0
print(c_len)


In [ ]:
length_of_options=[len(i) for i in response_options]


In [ ]:
def get_gpt(numChars,len_of_options, char_ratio,cost_in,cost_out):
  numC=[]
  numChars/=char_ratio
  numChars/= 1e6
  for i in len_of_options:
    fir = numChars*cost_in
    second = ((i/char_ratio * 1200))/(1e6)
    second*=cost_out
    numC.append(fir+second)
  return numC
ll=get_gpt(c_len,length_of_options,2,2.50,15.00)
print(ll)

In [ ]:
import numpy as np
np.mean(ll)

In [ ]:
get_gpt(c_len,length_of_options,2,2.50)

In [ ]:
# Select the columns, 'stack' them into one long list, and count
total_scale_counts = df.loc[:, 'Q1':'Q12'].stack().value_counts().sort_index()

print("Frequency of each response across ALL items:")
print(total_scale_counts)

In [ ]:
# Create a summary DataFrame
import pandas as pd
df = pd.read_csv("results.csv")
counts = df.loc[:, 'Q1':'Q12'].stack().value_counts().sort_index()
summary_df = pd.DataFrame({
    'Total Count': counts,
    'Percentage': (counts / counts.sum() * 100).round(2)
})
print(summary_df)

In [ ]:
counts

In [ ]:
import matplotlib.pyplot as plt

plt.hist(summary_df["Percentage"])

In [ ]:
import numpy as np
j=np.arange(0.1,2.05,0.05).tolist()
reverse_questions = [2, 4, 6, 14, 16, 17]
reverse_cols = ["Q2", "Q4", "Q6"]

for i in j:
  the_s=f"{i:.2f}_temp.csv"
  clean_csv(the_s)
  df=pd.read_csv(the_s)
  for col in reverse_cols:
    df[col] = df[col].map({1: 5, 2: 4, 3: 3, 4: 2, 5: 1})
  df.to_csv(the_s,index=False)